# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [1]:
import pandas as pd

# Cargar CSV
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Limpieza básica de columnas: espacios, minúsculas y guiones bajos
df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]

# Revisar los tipos de datos y columnas
print(df.head())
print(df.dtypes)


   unnamed:_0 customer       state  customer_lifetime_value response  \
0           0  DK49336     Arizona              4809.216960       No   
1           1  KX64629  California              2228.525238       No   
2           2  LZ68649  Washington             14947.917300       No   
3           3  XL78013      Oregon             22332.439460      Yes   
4           4  QA50777      Oregon              9025.067525       No   

   coverage education effective_to_date employmentstatus gender  ...  \
0     Basic   College           2/18/11         Employed      M  ...   
1     Basic   College           1/18/11       Unemployed      F  ...   
2     Basic  Bachelor           2/10/11         Employed      M  ...   
3  Extended   College           1/11/11         Employed      M  ...   
4   Premium  Bachelor           1/17/11    Medical Leave      F  ...   

   number_of_open_complaints number_of_policies     policy_type        policy  \
0                        0.0                  9  Corp

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [2]:
# Convertimos total_claim_amount a numérico por si acaso
df['total_claim_amount'] = pd.to_numeric(df['total_claim_amount'], errors='coerce')

# Filtrado
low_claim_yes_response = df[(df['total_claim_amount'] < 1000) & (df['response'] == "Yes")]

# Revisión
print(low_claim_yes_response.shape)
print(low_claim_yes_response.head())


(1399, 26)
    unnamed:_0 customer       state  customer_lifetime_value response  \
3            3  XL78013      Oregon             22332.439460      Yes   
8            8  FM55990  California              5989.773931      Yes   
15          15  CW49887  California              4626.801093      Yes   
19          19  NJ54277  California              3746.751625      Yes   
27          27  MQ68407      Oregon              4376.363592      Yes   

    coverage education effective_to_date employmentstatus gender  ...  \
3   Extended   College           1/11/11         Employed      M  ...   
8    Premium   College           1/19/11         Employed      M  ...   
15     Basic    Master           1/16/11         Employed      F  ...   
19  Extended   College           2/26/11         Employed      F  ...   
27   Premium  Bachelor           2/28/11         Employed      F  ...   

    number_of_open_complaints number_of_policies     policy_type  \
3                         0.0              

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [3]:
# Convertimos columnas necesarias a numérico
df['monthly_premium_auto'] = pd.to_numeric(df['monthly_premium_auto'], errors='coerce')
df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')

# Filtrar clientes con response "Yes"
yes_response = df[df['response'] == "Yes"]

# Agrupar por policy_type y gender
agg_stats = yes_response.groupby(['policy_type', 'gender']).agg({
    'monthly_premium_auto': 'mean',
    'customer_lifetime_value': 'mean',
    'total_claim_amount': 'mean'
}).reset_index()

print(agg_stats)


      policy_type gender  monthly_premium_auto  customer_lifetime_value  \
0  Corporate Auto      F             94.301775              7712.628736   
1  Corporate Auto      M             92.188312              7944.465414   
2   Personal Auto      F             98.998148              8339.791842   
3   Personal Auto      M             91.085821              7448.383281   
4    Special Auto      F             92.314286              7691.584111   
5    Special Auto      M             86.343750              8247.088702   

   total_claim_amount  
0          433.738499  
1          408.582459  
2          452.965929  
3          457.010178  
4          453.280164  
5          429.527942  


4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [4]:
# Contar clientes por estado
state_counts = df['state'].value_counts()

# Filtrar estados con más de 500 clientes
large_states = state_counts[state_counts > 500]
print(large_states)


state
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: count, dtype: int64


In [ ]:
# Estadísticas de customer lifetime value por educación y género
# Agrupar por education y gender
clv_stats = df.groupby(['education', 'gender'])['customer_lifetime_value'].agg(['max','min','median']).reset_index()

print(clv_stats)


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [5]:
# Convertimos la columna de fecha a datetime
df['effective_to_date'] = pd.to_datetime(df['effective_to_date'], errors='coerce')

# Extraer mes como nombre o número
df['month'] = df['effective_to_date'].dt.month_name()  # o .dt.month para número

# Contar número de pólizas por estado y mes
policy_counts = df.groupby(['state','month']).size().reset_index(name='num_policies')

# Encontrar los top 3 estados con más pólizas vendidas
top_states = df['state'].value_counts().head(3).index.tolist()

# Filtrar por top 3 estados
top_policy_counts = policy_counts[policy_counts['state'].isin(top_states)]

# Pivot para mostrar meses como columnas y estados como filas
pivot_table = top_policy_counts.pivot(index='state', columns='month', values='num_policies').fillna(0)

# Ordenar columnas por calendario si quieres
months_order = ['January','February','March','April','May','June','July','August','September','October','November','December']
pivot_table = pivot_table.reindex(columns=months_order)

print(pivot_table)


month       January  February  March  April  May  June  July  August  \
state                                                                  
Arizona        1008       929    NaN    NaN  NaN   NaN   NaN     NaN   
California     1918      1634    NaN    NaN  NaN   NaN   NaN     NaN   
Oregon         1565      1344    NaN    NaN  NaN   NaN   NaN     NaN   

month       September  October  November  December  
state                                               
Arizona           NaN      NaN       NaN       NaN  
California        NaN      NaN       NaN       NaN  
Oregon            NaN      NaN       NaN       NaN  


C:\Users\eduar\AppData\Local\Temp\ipykernel_26632\2384335123.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['effective_to_date'] = pd.to_datetime(df['effective_to_date'], errors='coerce')


6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

KeyError: "The following id_vars or value_vars are not present in the DataFrame: ['email', 'phone', 'web']"

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
# your code goes here